# 03 - Frozen VLM Encode + Train OfficialMiniVLDiT

Phase 1 encode video frame + language bằng frozen encoder. Primary là NVIDIA/Eagle; fallback SigLIP.
Phase 2 train DiT/action head từ scratch với cross-attention, action chunk H=16, Beta timestep, masked flow-matching loss.

In [ ]:
!pip install -q pandas pyarrow numpy tqdm matplotlib pillow transformers accelerate av opencv-python

In [ ]:
from pathlib import Path
import json, random, time
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.distributions import Beta

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available(): print("gpu:", torch.cuda.get_device_name(0))

SMOKE_TEST = False
ACTION_HORIZON = 16
STATE_HORIZON = 1
MAX_ENCODE_TRAIN_SAMPLES = 512 if SMOKE_TEST else None
MAX_ENCODE_TEST_SAMPLES = 256 if SMOKE_TEST else None
MAX_TRAIN_SAMPLES = 2048 if SMOKE_TEST else None
MAX_STEPS_PER_EPOCH = 20 if SMOKE_TEST else None
EPOCHS = 1; BATCH_SIZE = 256; NUM_WORKERS = 2
LR = 1e-4; WEIGHT_DECAY = 1e-5; USE_AMP = True
HIDDEN_DIM = 512; NUM_LAYERS = 8; NUM_HEADS = 8; DROPOUT = 0.1
PRIMARY_ENCODER = "nvidia/Eagle-Block2A-2B-v2"
FALLBACK_ENCODER = "google/siglip-base-patch16-224"
FORCE_FALLBACK = False
OUTPUT_ROOT = Path("/kaggle/working/gr00t_official_mini_runs"); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_NAME = f"official_mini_vldit_H{ACTION_HORIZON}"

In [ ]:
def find_dir(name, required):
    cand = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if base.exists(): cand.extend(base.rglob(name))
    for c in sorted(cand, key=str):
        if c.is_dir() and all((c / r).exists() for r in required): return c
    raise FileNotFoundError(f"Không tìm thấy {name}")

PREPARED_ROOT = find_dir(f"gr00t_prepared_official_H{ACTION_HORIZON}", ["states_train.npy", "actions_train_chunk.npy", "action_mask_train.npy", "video_frame_manifest_train.parquet", "normalization_stats.json", "prepare_report.json"])
stats = json.loads((PREPARED_ROOT / "normalization_stats.json").read_text())
prepare_report = json.loads((PREPARED_ROOT / "prepare_report.json").read_text())
print("PREPARED_ROOT:", PREPARED_ROOT)
print(json.dumps({k: prepare_report[k] for k in ["state_horizon", "action_horizon", "train_samples", "test_samples"]}, indent=2))

In [ ]:
def decode_frame_pyav(video_path, frame_index):
    import av
    container = av.open(video_path)
    try:
        stream = container.streams.video[0]
        last = None
        for i, frame in enumerate(container.decode(stream)):
            last = frame
            if i >= frame_index:
                return frame.to_image().convert("RGB")
        if last is None: raise RuntimeError("empty video")
        return last.to_image().convert("RGB")
    finally:
        container.close()

def decode_frame_cv2(video_path, frame_index):
    import cv2
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): raise RuntimeError("cv2 cannot open video")
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
    ok, frame = cap.read()
    if not ok:
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, total - 1)); ok, frame = cap.read()
    cap.release()
    if not ok: raise RuntimeError("cv2 cannot decode frame")
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

def decode_frame(video_path, frame_index):
    # Uu tien PyAV vi doc video on dinh; neu loi thi fallback sang OpenCV.
    try: return decode_frame_pyav(video_path, int(frame_index))
    except Exception: return decode_frame_cv2(video_path, int(frame_index))

In [ ]:
from transformers import AutoModel, AutoProcessor

class FrozenVLEncoder:
    def __init__(self, primary_id, fallback_id, force_fallback=False):
        self.primary_id = primary_id; self.fallback_id = fallback_id; self.force_fallback = force_fallback
        self.encoder_name = None; self.fallback_used = False; self.processor = None; self.model = None
        self.feature_dim = None; self.num_tokens = 2
        self.load()
    def try_load(self, model_id):
        proc = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        model = AutoModel.from_pretrained(model_id, trust_remote_code=True, low_cpu_mem_usage=True,
                                           torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
        model.eval().to(device)
        for p in model.parameters(): p.requires_grad_(False)
        return proc, model
    def load(self):
        if not self.force_fallback:
            try:
                self.processor, self.model = self.try_load(self.primary_id)
                self.encoder_name = self.primary_id
                print("Loaded primary encoder:", self.encoder_name)
                return
            except Exception as exc:
                print("WARN primary failed, fallback:", repr(exc))
        self.processor, self.model = self.try_load(self.fallback_id)
        self.encoder_name = self.fallback_id; self.fallback_used = True
        print("Loaded fallback encoder:", self.encoder_name)
    @torch.no_grad()
    def encode_batch(self, images, texts):
        # Encoder duoc freeze hoan toan; Notebook 03 chi train DiT/action head tu scratch.
        inputs = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
        with torch.cuda.amp.autocast(enabled=USE_AMP and torch.cuda.is_available()):
            if hasattr(self.model, "get_image_features") and hasattr(self.model, "get_text_features"):
                image_feat = self.model.get_image_features(pixel_values=inputs["pixel_values"])
                text_keys = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
                text_feat = self.model.get_text_features(**text_keys) if text_keys else torch.zeros_like(image_feat)
                feats = torch.stack([F.normalize(image_feat.float(), dim=-1), F.normalize(text_feat.float(), dim=-1)], 1)
            else:
                out = self.model(**inputs, output_hidden_states=True, return_dict=True)
                hidden = getattr(out, "last_hidden_state", None)
                if hidden is None:
                    hidden = out.hidden_states[-1]
                pooled = hidden.float().mean(1)
                feats = torch.stack([pooled, pooled], 1)
        self.feature_dim = int(feats.shape[-1]); self.num_tokens = int(feats.shape[1])
        return feats.cpu().numpy().astype(np.float16)

In [ ]:
def select_manifest(split, max_samples):
    df = pd.read_parquet(PREPARED_ROOT / f"video_frame_manifest_{split}.parquet")
    df = df[(df.has_video == True) & (df.video_path.astype(str).str.len() > 0)].copy()
    if df.empty: raise RuntimeError(f"No video-backed samples for {split}")
    df = df.sort_values(["subset", "episode_index", "frame_index"]).reset_index(drop=True)
    if max_samples is not None and len(df) > max_samples:
        df = df.sample(n=int(max_samples), random_state=SEED).sort_values("sample_id").reset_index(drop=True)
    return df

def encode_split(split, max_samples, encoder, batch_size=16):
    feat_path = OUTPUT_ROOT / f"vl_features_{split}.npy"
    idx_path = OUTPUT_ROOT / f"vl_feature_index_{split}.parquet"
    if feat_path.exists() and idx_path.exists():
        return feat_path, idx_path, {"cached": True, "shape": list(np.load(feat_path, mmap_mode="r").shape)}
    manifest = select_manifest(split, max_samples)
    chunks, rows, failed = [], [], []
    for start in tqdm(range(0, len(manifest), batch_size), desc=f"encode {split}"):
        batch = manifest.iloc[start:start+batch_size]
        images, texts, ok = [], [], []
        for _, r in batch.iterrows():
            try:
                images.append(decode_frame(r.video_path, int(r.frame_index)))
                texts.append(str(r.task_text) if str(r.task_text) else "perform the manipulation task")
                ok.append(r)
            except Exception as exc:
                failed.append({"sample_id": int(r.sample_id), "error": str(exc)})
        if not images: continue
        feats = encoder.encode_batch(images, texts); base = sum(c.shape[0] for c in chunks); chunks.append(feats)
        for j, r in enumerate(ok):
            rows.append({"feature_index": int(base + j), "sample_id": int(r.sample_id), "subset": r.subset, "episode_index": int(r.episode_index), "frame_index": int(r.frame_index), "task_text": str(r.task_text)})
    features = np.concatenate(chunks, 0)
    np.save(feat_path, features); pd.DataFrame(rows).to_parquet(idx_path, index=False)
    report = {"split": split, "requested": int(len(manifest)), "encoded": int(features.shape[0]), "failed": int(len(failed)),
              "feature_shape": list(features.shape), "encoder_name": encoder.encoder_name, "fallback_used": encoder.fallback_used, "failed_examples": failed[:20]}
    (OUTPUT_ROOT / f"video_decode_report_{split}.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return feat_path, idx_path, report

encoder = FrozenVLEncoder(PRIMARY_ENCODER, FALLBACK_ENCODER, FORCE_FALLBACK)
train_feat, train_idx, train_enc_report = encode_split("train", MAX_ENCODE_TRAIN_SAMPLES, encoder)
test_feat, test_idx, test_enc_report = encode_split("test", MAX_ENCODE_TEST_SAMPLES, encoder)
print(train_enc_report); print(test_enc_report)

In [ ]:
class OfficialDataset(Dataset):
    def __init__(self, split, feat_path, idx_path, max_samples=None):
        self.split = split
        self.states = np.load(PREPARED_ROOT / f"states_{split}.npy", mmap_mode="r")
        self.actions = np.load(PREPARED_ROOT / f"actions_{split}_chunk.npy", mmap_mode="r")
        self.masks = np.load(PREPARED_ROOT / f"action_mask_{split}.npy", mmap_mode="r")
        self.features = np.load(feat_path, mmap_mode="r")
        self.index = pd.read_parquet(idx_path).sort_values("feature_index").reset_index(drop=True)
        if max_samples is not None and len(self.index) > max_samples:
            self.index = self.index.sample(n=int(max_samples), random_state=SEED).reset_index(drop=True)
        self.sm = np.asarray(stats["state_mean"], np.float32); self.ss = np.asarray(stats["state_std"], np.float32)
        self.am = np.asarray(stats["action_mean"], np.float32); self.asd = np.asarray(stats["action_std"], np.float32)
    def __len__(self): return len(self.index)
    def __getitem__(self, i):
        r = self.index.iloc[i]; sid = int(r.sample_id); fid = int(r.feature_index)
        # Normalize bang thong ke train split; Notebook 04 se denormalize lai khi tinh metric raw.
        state = (np.asarray(self.states[sid], np.float32) - self.sm) / self.ss
        action = (np.asarray(self.actions[sid], np.float32) - self.am) / self.asd
        return {"state": torch.from_numpy(state), "action": torch.from_numpy(action),
                "mask": torch.from_numpy(np.asarray(self.masks[sid], np.float32)),
                "vl": torch.from_numpy(np.asarray(self.features[fid], np.float32)), "sample_id": sid}
def collate(batch):
    return {k: torch.stack([b[k] for b in batch]) if k != "sample_id" else torch.tensor([b[k] for b in batch]) for k in batch[0]}

train_ds = OfficialDataset("train", train_feat, train_idx, MAX_TRAIN_SAMPLES)
test_ds = OfficialDataset("test", test_feat, test_idx, None)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=True, collate_fn=collate)
FEATURE_DIM = int(train_ds.features.shape[-1]); NUM_TOKENS = int(train_ds.features.shape[1])
print("train_samples:", len(train_ds), "FEATURE_DIM:", FEATURE_DIM, "tokens:", NUM_TOKENS)

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, hidden_dim, buckets=1000):
        super().__init__()
        self.buckets = buckets
        self.embed = nn.Embedding(buckets, hidden_dim)
        self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(hidden_dim, hidden_dim))
    def forward(self, t):
        idx = torch.clamp((t * self.buckets).long(), 0, self.buckets - 1)
        return self.mlp(self.embed(idx))

class CrossBlock(nn.Module):
    def __init__(self, hidden_dim, heads, dropout):
        super().__init__()
        self.n1 = nn.LayerNorm(hidden_dim)
        self.sa = nn.MultiheadAttention(hidden_dim, heads, dropout=dropout, batch_first=True)
        self.n2 = nn.LayerNorm(hidden_dim)
        self.ca = nn.MultiheadAttention(hidden_dim, heads, dropout=dropout, batch_first=True)
        self.n3 = nn.LayerNorm(hidden_dim)
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim), nn.Dropout(dropout)
        )
    def forward(self, x, mem):
        y = self.n1(x)
        x = x + self.sa(y, y, y, need_weights=False)[0]
        y = self.n2(x)
        x = x + self.ca(y, mem, mem, need_weights=False)[0]
        return x + self.ff(self.n3(x))

class OfficialMiniVLDiT(nn.Module):
    def __init__(self, state_dim, action_dim, state_horizon, action_horizon, vl_dim,
                 hidden_dim=512, layers=8, heads=8, dropout=0.1, buckets=1000):
        super().__init__()
        self.state_horizon = state_horizon
        self.action_horizon = action_horizon
        self.state_proj = nn.Linear(state_dim, hidden_dim)
        self.action_proj = nn.Linear(action_dim, hidden_dim)
        self.vl_proj = nn.Linear(vl_dim, hidden_dim)
        self.time = TimeEmbedding(hidden_dim, buckets)
        self.state_pos = nn.Embedding(state_horizon, hidden_dim)
        self.action_pos = nn.Embedding(action_horizon, hidden_dim)
        self.blocks = nn.ModuleList([CrossBlock(hidden_dim, heads, dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(hidden_dim)
        self.out = nn.Linear(hidden_dim, action_dim)
    def forward(self, noisy_action, state_history, vl_features, t):
        s_pos = self.state_pos(torch.arange(self.state_horizon, device=noisy_action.device))[None]
        a_pos = self.action_pos(torch.arange(self.action_horizon, device=noisy_action.device))[None]
        s = self.state_proj(state_history) + s_pos
        a = self.action_proj(noisy_action) + a_pos
        x = torch.cat([s, a], dim=1) + self.time(t)[:, None, :]
        mem = self.vl_proj(vl_features)
        for block in self.blocks:
            x = block(x, mem)
        return self.out(self.norm(x[:, self.state_horizon:]))

model = OfficialMiniVLDiT(44, 44, STATE_HORIZON, ACTION_HORIZON, FEATURE_DIM, HIDDEN_DIM, NUM_LAYERS, NUM_HEADS, DROPOUT).to(device)
num_params = sum(p.numel() for p in model.parameters())
print("params:", num_params)

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.95, 0.999), eps=1e-8)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and torch.cuda.is_available())
beta = Beta(torch.tensor(1.5, device=device), torch.tensor(1.0, device=device))
logs, global_step, start = [], 0, time.time()
model.train()
for epoch in range(1, EPOCHS + 1):
    max_steps = len(train_loader) if MAX_STEPS_PER_EPOCH is None else min(MAX_STEPS_PER_EPOCH, len(train_loader))
    for step, b in enumerate(tqdm(train_loader, total=max_steps, desc=f"epoch {epoch}/{EPOCHS}"), 1):
        if step > max_steps: break
        state = b["state"].to(device); action = b["action"].to(device); mask = b["mask"].to(device); vl = b["vl"].to(device)
        t = beta.sample((action.size(0),)).to(device=device, dtype=action.dtype)
        noise = torch.randn_like(action)
        noisy = (1 - t[:, None, None]) * noise + t[:, None, None] * action
        # Flow matching dung huong official: velocity = action - noise.
        target = action - noise
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP and torch.cuda.is_available()):
            pred = model(noisy, state, vl, t)
            loss = ((pred - target).pow(2) * mask).sum() / (mask.sum() + 1e-6)
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
        global_step += 1
        if global_step == 1 or global_step % 100 == 0:
            logs.append({"epoch": epoch, "step": step, "global_step": global_step, "loss": float(loss.detach().cpu()), "elapsed_sec": round(time.time() - start, 2)})

run_dir = OUTPUT_ROOT / RUN_NAME; run_dir.mkdir(parents=True, exist_ok=True)
cfg = {"model_name": "OfficialMiniVLDiT", "state_horizon": STATE_HORIZON, "action_horizon": ACTION_HORIZON, "state_dim": 44, "action_dim": 44,
       "vl_feature_dim": FEATURE_DIM, "vl_num_tokens": NUM_TOKENS, "hidden_dim": HIDDEN_DIM, "num_layers": NUM_LAYERS, "num_heads": NUM_HEADS,
       "dropout": DROPOUT, "num_timestep_buckets": 1000, "timestep_sampling": "Beta(1.5,1.0)", "target_velocity_sign": "action_minus_noise",
       "masked_loss": True, "learning_rate": LR, "batch_size": BATCH_SIZE, "epochs": EPOCHS, "num_params": int(num_params),
       "encoder_name": encoder.encoder_name, "fallback_used": encoder.fallback_used, "prepared_root": str(PREPARED_ROOT)}
pd.DataFrame(logs).to_csv(run_dir / "train_log.csv", index=False)
(run_dir / "config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
torch.save({"model_state_dict": model.state_dict(), "config": cfg, "normalization": stats, "global_step": global_step}, run_dir / "checkpoint_last.pt")
summary = {"status": "completed", "run_dir": str(run_dir), "full_epoch_completed": MAX_TRAIN_SAMPLES is None and MAX_STEPS_PER_EPOCH is None,
           "global_steps": int(global_step), "train_samples": int(len(train_ds)), "last_loss": logs[-1]["loss"] if logs else None,
           "elapsed_sec": round(time.time() - start, 2), "acceptance": {"action_horizon": ACTION_HORIZON, "state_horizon": STATE_HORIZON,
           "uses_vl_features": True, "cross_attention_dim": FEATURE_DIM, "masked_loss": True, "target_velocity_sign": "action_minus_noise"}}
(run_dir / "train_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(run_dir / "encode_report.json").write_text(json.dumps({"train": train_enc_report, "test": test_enc_report, "encoder_name": encoder.encoder_name, "fallback_used": encoder.fallback_used}, indent=2), encoding="utf-8")
if logs:
    df = pd.DataFrame(logs); plt.figure(figsize=(8,4)); plt.plot(df.global_step, df.loss); plt.grid(True, alpha=.3); plt.xlabel("step"); plt.ylabel("masked flow loss"); plt.tight_layout(); plt.savefig(run_dir / "loss_curve.png", dpi=160); plt.show()
print(json.dumps(summary, indent=2))
for p in sorted(run_dir.iterdir()): print("-", p.name, round(p.stat().st_size / (1024 ** 2), 3), "MB")